<a href="https://colab.research.google.com/github/MathiasGarnier/Algorithms-All---Old/blob/master/code/3_bert_training/Hands_on_3_SS.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Hands-On 3: Semantic Shifts


<p align="center">
  <img alt="Widnes Examiner (1883)" src="https://panoptes-uploads.zooniverse.org/subject_location/421af65f-ab5d-4977-9186-65d7f5938902.png" width="54%">
&nbsp; &nbsp; &nbsp; &nbsp;
  <img alt="Widnes Examiner (1916)" src="https://panoptes-uploads.zooniverse.org/subject_location/775591a5-87a9-426d-af8c-08f130a397d8.png" width="41%">
</p>

Let's explore diachronic vector representations, using OCRed historical newspaper excerpts from the [Living with Machines program](https://livingwithmachines.ac.uk/).

Learn more on [Semantic Change on Wikipedia](https://en.wikipedia.org/wiki/Semantic_change).


In [1]:
#@title A glimpse at the data:


from IPython.core.display import display, HTML
import urllib.request

fp = urllib.request.urlopen("https://raw.githubusercontent.com/d-noe/NLP_DH_F26/refs/heads/main/props/lwm_advanced_trolley.html")
mybytes = fp.read()

mystr = mybytes.decode("utf8")
fp.close()

display(HTML(mystr))

## Set-up

Feel free to jump directly to the next section if you run the notebook locally and that you already have a ready-to-use environment.

Install and import necessary Python libraries and modules.

This notebook will mainly rely on [`transformers` Python library](https://huggingface.co/docs/transformers/installation), we will also use [`scikit-learn`](https://scikit-learn.org/stable/install.html) for dimensionality reduction, and [`altair`](https://altair-viz.github.io/) for visualisation.

In [2]:
! pip install transformers

In [3]:
# For BERT -> use lighter version DistilBert
from transformers import DistilBertTokenizerFast, DistilBertModel

# For data manipulation and analysis
import pandas as pd
pd.options.display.max_colwidth = 200
import numpy as np
from sklearn.decomposition import PCA

# For interactive data visualization
import altair as alt

In [4]:
# are 'GPU's available?
# If using Colab, you can change your runtime to access GPUs
# but it is not really needed here, computations are not too long on CPUs
import torch
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(f"Device: {DEVICE}")

Device: cuda


## Familiarize with the data


In the context of this experiment, the data (originally from [here](https://livingwithmachines.ac.uk/)) is presented in the `.csv`, which can be conviniently loaded as a `pandas.DataFrame`. This file contains the following columns:
- `text` [string]: Historical Newspaper article excerpt (OCRed)
- `target` [string]: the word of interest `text` (example: *'trolley'*)
- ...

**Propositions**

- Select a word from the following list: `trolley`, `car`, `coach`, `bike`.
- Filter the dataset to contain only rows related to the selected word `target`
- Have a look at the data:
  - What are texts like?
  - What is the time distribution of the included articles?
  - Where are these articles from?
  - What are the annotation labels?
- Find a way to filter down the data
  - If dealing with technical/computational limitations: reduce the number of samples to limit the computation costs
  - Reflect on the impact that it could have during downstream analysis and keep this in mind
  - Plus, here we have the chance to have annotated data (see `annot` column), why not use the annotations?


In [16]:
!wget https://raw.githubusercontent.com/d-noe/NLP_DH_F26/refs/heads/main/code/scripts/helpers.py
from helpers import load_csv_from_github

# Load the data
df = load_csv_from_github("data/word_sense/lwm_annot.csv")
print(f"The DataFrame contains {len(df)} rows.")

# Display 5 random rows
df.sample(5)

--2026-10-14 09:36:07--  https://raw.githubusercontent.com/d-noe/NLP_DH_F26/refs/heads/main/code/scripts/helpers.py
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.109.133, 185.199.108.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.109.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 4787 (4.7K) [text/plain]
Saving to: ‘helpers.py.4’

helpers.py.4        100%[===================>]   4.67K  --.-KB/s    in 0s      

2026-10-14 09:36:07 (77.7 MB/s) - ‘helpers.py.4’ saved [4787/4787]

The DataFrame contains 3176 rows.


,text,annot,newspaper_date,newspaper_title,newspaper_place,image_url,image_url_conflict_values,target,year
2488,"RUDGE-MULTI Combination, Phoenix el coach built side-car, 1913; 490. cash.—Fin ney's Drug Stores, Wath-on-Dearne. C 740 TN STOOK. Rbyal 'Enfield OXCLFAS, made I like a gun, from 13 Gas. All models...",The image doesn't contain enough information to assess how the word 'coach' is used,1920-08-21,"Penistone, Stocksbridge and Hoyland Express, etc.","Penistone, South Yorkshire, England",https://panoptes-uploads.zooniverse.org/subject_location/4cc512b1-3417-4346-8abb-807def3359c5.png,NaN,coach,1920
2107,"FLEETWOOD POLIC 11 N SW S. —SUSEIR Good fellow has been (*omitted to goal for 14 days ""hard"" for being drunk and disorderly in Poulton Street. Archibald Jemiaon, and Edward Wingett, for similar of...",The image doesn't contain enough information to assess how the word 'coach' is used,1893-06-30,The Blackpool Herald.,"Blackpool, Lancashire, England",https://panoptes-uploads.zooniverse.org/subject_location/985809b1-9906-406e-af2e-bc4b05bb7794.png,NaN,coach,1893
2511,"SMITH & SONS, Auctioneers and Estate Agents, 3, Arole-street, Birkenhead, base the following fIOUA'S to LKT WTON.—Countri . 1110-IDENCE, within easy ac- CCM ef the el y, standing upon ha'f an acre...",A horse-drawn private or public carriage for carrying passengers,1882-08-12,The Birkenhead News and Wirral General Advertiser.,"Birkenhead, Merseyside, England",https://panoptes-uploads.zooniverse.org/subject_location/409aaa08-d6ea-4d72-86a1-febe36f30664.png,NaN,coach,1882
230,In the course of his lecture at the Tyne Theatre last night. Dr Andrew Wilson dealt rather severely with the pretensions of the Duke of Argyll in the world of science. He remarkal that the Duke ha...,"A railway carriage or wagon, usually powered by steam",1897-12-06,The Northern Guardian.,"Hartlepool, Cleveland, England",https://panoptes-uploads.zooniverse.org/subject_location/a139215e-223a-4588-9a1d-1c814cdeb870.png,"['https://panoptes-uploads.zooniverse.org/subject_location/a139215e-223a-4588-9a1d-1c814cdeb870.png', 'https://panoptes-uploads.zooniverse.org/subject_location/a9a01b07-fc6b-48ec-977f-63b61937ebd8...",car,1897
1451,"The Park Avenue trolley line in the city of Rochester is crossed by three consecutive streets which bear masculine surnames. An Irishman with a carpet bag entered one of the cars the other day, an...",A trolley-car powered by steam or electricity,1898-06-17,The Cotton Factory Times.,"Manchester, Greater Manchester, England",https://panoptes-uploads.zooniverse.org/subject_location/d6e09a52-8b20-465e-a75e-58a454f7393c.png,NaN,trolley,1898


In [ ]:
# TODO: run you code here, feel free to add cells
  # - Select a word from the following list: `trolley`, `car`, `coach`, `bike`.
  # - Filter the dataset to contain only rows related to the selected word `target`
# - Have a look at the data:
#   - What are texts like?
#   - What is the time distribution of the included articles?
#   - Where are these articles from?
#   - What are the annotation labels?
# - Find a way to filter down the data
#   - If dealing with technical/computational limitations: reduce the number of samples to limit the computation costs
#   - Reflect on the impact that it could have during downstream analysis and keep this in mind
#   - Plus, here we have the chance to have annotated data (see `annot` column), why not use the annotations?


In [20]:
word = "trolley"
prefiltered_df = df[df["annot"].str.contains(word)]
# prefiltered_df

df_newspaper = prefiltered_df.groupby(["newspaper_title"])
df_sorted_dates = prefiltered_df.sort_values(by="newspaper_date")

In [22]:
# df_newspaper.head()
df_sorted_dates

,text,annot,newspaper_date,newspaper_title,newspaper_place,image_url,image_url_conflict_values,target,year
1413,"TO CARMEN.—Mr. ANDREW HIND has for SALE a capital BUSINESS, returning £4'► per week. Stabling, yard, and country house. Rent, £25. Price, including eight horses, vans, cart, trolley, harness, ctc....",The image doesn't contain enough information to assess how the word 'trolley' is used in this context,1864-03-28,The Sun.,"London, England",https://panoptes-uploads.zooniverse.org/subject_location/1d034bb5-7915-43ac-84b6-849ce3fe28bf.png,NaN,trolley,1864
1420,"TO CARMEN.—Mr. ANDREW HIND has for SALE a capital BUSINESS, returning £4l per week. Stabling, yard, and country house. Rent, £25. Price, including eight horses, vans, cart, trolley, harness, £350....",The image doesn't contain enough information to assess how the word 'trolley' is used in this context,1864-04-02,The Sun.,"London, England",https://panoptes-uploads.zooniverse.org/subject_location/11cddd4d-c16d-4e53-af37-fcdb703ad5d6.png,NaN,trolley,1864
1414,"TO CARMEN.—Mr. ANDREW HIND has for SALE a capital BUSINESS, returning £4) per week. Stabling, yard, and country house. Rent, £25. Price, including eight horses, vans, cart, trolley, harness, &c., ...",The image doesn't contain enough information to assess how the word 'trolley' is used in this context,1864-04-11,The Sun.,"London, England",https://panoptes-uploads.zooniverse.org/subject_location/cd5f7004-979b-4d26-87f0-dced119d3c29.png,NaN,trolley,1864
1056,"A fatal I.,o3itlent occurred on the above line on Wednesday, 'to a young Man named IZA,bert Vincent, in the employ of Messrs Miller and Co. It appears that the poor fellow was with others, engaged...",A trolley-car powered by steam or electricity,1874-10-16,"The Central Glamorgan Gazette, and General, Commercial, and Agricultural Advertiser.","Bridgend, Mid Glamorgan, Wales",https://panoptes-uploads.zooniverse.org/subject_location/48ac2036-5abc-4de2-807c-16d175938aea.png,"['https://panoptes-uploads.zooniverse.org/subject_location/48ac2036-5abc-4de2-807c-16d175938aea.png', 'https://panoptes-uploads.zooniverse.org/subject_location/0284166c-1141-469b-9afd-1db9e74c6ed6...",trolley,1874
1600,"were marks Of white lead on the trolley, which were not there when it was taken. It was also broken. The prisoner was —... . . .",The image doesn't contain enough information to assess how the word 'trolley' is used in this context,1875-04-10,The Cannock Chase Examiner.,"Cannock, Staffordshire, England",https://panoptes-uploads.zooniverse.org/subject_location/c2f9ed2d-6c04-4eb1-bbca-ceb22b4804ff.png,NaN,trolley,1875
...,...,...,...,...,...,...,...,...,...
1540,"William Glover. the driver of the ear on which the deceased was conductrtss. said that ho had had 15 or 16 years' es. perienee as an electric tramcar driver, and had never before had a fatal accid...",A trolley-car powered by steam or electricity,1918-05-15,The Birkenhead News and Wirral General Advertiser.,"Birkenhead, Merseyside, England",https://panoptes-uploads.zooniverse.org/subject_location/c70c5116-be79-467a-b0f6-c4a27015ba52.png,NaN,trolley,1918
1046,"A Unrporafien tramway drtver named Oarnernn v.et a terrible death at the Palm,* depot Ile left ute ....eh teeter on hie car while he waa placing' thn trolley on the vire. The car moving forward ri...",A trolley-car powered by steam or electricity,1918-09-21,The Haslingden Gazette.,"Haslingden, Lancashire, England",https://panoptes-uploads.zooniverse.org/subject_location/af4243c3-a0ae-4bb4-8eda-72e5a8151694.png,NaN,trolley,1918
1439,"The Birkenhead Corporation was me defendant in an action commenced on Wednesday, in the Civil Court of the Liverpool Assizes, before Mr. Justice Greer. It was in the nature of a sequel to a tramwa...",A trolley-car powered by steam or electricity,1919-05-10,The Birkenhead News and Wirral General Advertiser.,"Birkenhead, Merseyside, England",https://panoptes-uploads.zooniverse.org/subject_location/050f12ea-d82e-4c3a-b1af-c056000f0e85.

In [ ]:
# By now, you should have filtered down the `df` to the samples you want to analyze.
# Let's store it in `df_filtered`

# Un-comment and store it here:
# df_filtered =

## Produce Vector Representations

Now, that you are all set with your data, let's focus on the core of this approach: producing embeddings from the data using a pre-trained LM.

First load the model and tokenizer, then produce the vectors and retrieve the vectors associated with the word that you are studying.

Earlier, in the tutorial, we used a quick and dirty heuristic to retrieve the vector representations associated with our focus word. That was not very convincing nor robust... Let's address this shortcoming: define a `get_vector_representations` function that can extract the embeddings for any word!

In [23]:
tokenizer = DistilBertTokenizerFast.from_pretrained('distilbert-base-uncased')
model = DistilBertModel.from_pretrained("distilbert-base-uncased").to(DEVICE)

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  268MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertModel LOAD REPORT from: distilbert-base-uncased
Key                     | Status     |  | 
------------------------+------------+--+-
vocab_projector.bias    | UNEXPECTED |  | 
vocab_layer_norm.bias   | UNEXPECTED |  | 
vocab_transform.bias    | UNEXPECTED |  | 
vocab_layer_norm.weight | UNEXPECTED |  | 
vocab_transform.weight  | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [39]:
ancien_truc = """
field_token_pids = [np.argmax(t.cpu().numpy()==word_id) for t in tokenized_sentences["input_ids"]]

field_vectors = np.array([
    o[p_id].detach().cpu().numpy()
    for o, p_id in zip(outputs["last_hidden_state"], field_token_pids)
])
"""


def get_vector_representations(
    model,
    tokenizer,
    texts:list[str],
    focus_word:str,
    **kwargs
):
  """
  Input
    - model:
        pre-trained LM, used to embed texts
    - tokenizer:
        associated tokenizer
    - texts: [list[str]]
        texts to be embedded
    - focus_word: [str]
        word of interest: get the representations at this position
  Output
    - embeddings: [array-like]
        embeddings for the focus_word in each instance of 'texts'
  """


  # compris à l'envers.
  tokenized_sentences = tokenizer(
    texts,
    truncation=True,
    padding=True,
    return_tensors="pt"
  ).to(DEVICE)

  inputs = tokenizer.tokenize(focus_word)
  word_id = tokenizer.convert_tokens_to_ids(inputs)

  field_token_pids = [np.argmax(t.cpu().numpy()==word_id) for t in tokenized_sentences["input_ids"]]


  # raise NotImplementedError("You need to implement the 'get_vector_representations' function first!")
  return field_token_pids

In [40]:
list_of_texts = prefiltered_df["text"].to_list()
list_of_texts

['! The Liverpool electric trams are worked on the trolley system, and where overhead electric telephone wires cross there is supposed to lie 1 over the top of the win, a non.conductor, so that in the event of either the telegraph or telephone wires snapping there would be no contact. On Monday night these precautions did not prevent a telephone wire which snapped from coming in contact with the trolley-wire on the tvertree tram route. The result was that the ken telethon,. wit, tbrained contact, and played havoc, creating dismay among the outside tram passenger, On,. luau was struck by the live telephone wire, which encircled his leg, and in a moment lie wan a corpse. Two other persons wore also struck, and sustained injuries neces sitating their removal to the Pifirmary. The whole electric system of the city was at once stopped. A second death iiertirred through a coal eastern horse being caught by the wire. The driver went to the assistance of the animal, sad both were killed.',
 '"

In [41]:
embeddings = get_vector_representations(
    model=model,
    tokenizer=tokenizer,
    texts=list_of_texts,
    focus_word="engine",
)

In [42]:
embeddings

[np.int64(0),
 np.int64(0),
 np.int64(0),
 np.int64(0),
 np.int64(0),
 np.int64(0),
 np.int64(0),
 np.int64(0),
 np.int64(0),
 np.int64(0),
 np.int64(0),
 np.int64(0),
 np.int64(0),
 np.int64(0),
 np.int64(0),
 np.int64(0),
 np.int64(0),
 np.int64(0),
 np.int64(0),
 np.int64(0),
 np.int64(0),
 np.int64(0),
 np.int64(0),
 np.int64(0),
 np.int64(0),
 np.int64(0),
 np.int64(0),
 np.int64(0),
 np.int64(0),
 np.int64(0),
 np.int64(0),
 np.int64(0),
 np.int64(0),
 np.int64(0),
 np.int64(0),
 np.int64(0),
 np.int64(261),
 np.int64(0),
 np.int64(0),
 np.int64(0),
 np.int64(0),
 np.int64(0),
 np.int64(0),
 np.int64(0),
 np.int64(0),
 np.int64(0),
 np.int64(0),
 np.int64(0),
 np.int64(0),
 np.int64(0),
 np.int64(0),
 np.int64(0),
 np.int64(0),
 np.int64(0),
 np.int64(0),
 np.int64(0),
 np.int64(0),
 np.int64(17),
 np.int64(0),
 np.int64(0),
 np.int64(0),
 np.int64(0),
 np.int64(125),
 np.int64(0),
 np.int64(0),
 np.int64(0),
 np.int64(0),
 np.int64(0),
 np.int64(0),
 np.int64(0),
 np.int64(0),
 

### Visualise your results

Apply dimensionality reduction on the embeddings, and visualize their distribution in a 2D space.

In [ ]:
# perform dimensionality reduction on `embeddings`
# + store in a dataframe to be able to visualize directly below

# 1. Compute 'reduced_vectors'




In [ ]:
# 2. Store in a DataFrame

# df_plot = df_filtered.copy()
# df_plot["dim_1"] = reduced_vectors[:,0]
# df_plot["dim_2"] = reduced_vectors[:,1]

In [ ]:
chart = alt.Chart(
    df_plot,
    title=f"Word PCA Distribution: {focus_word}"
).mark_circle(size=200).encode(
    alt.X('dim_1',
        scale=alt.Scale(zero=False)
    ),
    y="dim_2",
    color= "label_sense",
    tooltip=['target', 'label_sense', 'text'],
    ).interactive().properties(
    width=500,
    height=500
)
chart

### Analyse

Ideas:

- Find if their is really a *drift* within the representation space.
- Recall last week's semantic shift analysis with `Word2Vec`, how is that approach different? How do the results compare?



## Want more?

**Challenges:**
1. Can you find an embedding dimension that encodes time across concepts?
2. Explore different models (other than `distilbert`), how does it impact the results?
3. We used the "last hidden state" vector at the position of the `focus_word`, is this the best possible choice? Try to extract different vector representations (e.g.; from other layer(s), using mean-pooling, or concatenation) and study the influence on the visualisation / analysis.